In [1]:
import sqlite3
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
#Load environment variable
load_dotenv()

True

In [3]:
#Define the correct relative path
data_dir = os.path.join("..", "data", "cleaned")  #Location of cleaned data
merged_dir = os.path.join("..", "data", "merged")  #New folder for merged data
os.makedirs(merged_dir, exist_ok=True)  #Ensure 'merged' directory exists

In [4]:
#Define file paths
reddit_file = os.path.join(data_dir, "RedditComments_Cleaned.csv")
news_file = os.path.join(data_dir, "GoogleNews_Cleaned.csv")
youtube_file = os.path.join(data_dir, "YoutubeComments_Cleaned.csv")
output_file = os.path.join(merged_dir, "merged_data.csv")  #Ensure output is saved in 'merged/' folder

In [5]:
#Safely load CSV files
def load_csv(file_path):
    if not os.path.exists(file_path):
        print(f"⚠️ Warning: {file_path} not found. Skipping...")
        return None
    try:
        return pd.read_csv(file_path)
    except pd.errors.EmptyDataError:
        print(f"⚠️ Warning: {file_path} is empty. Skipping...")
        return None
    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        return None

In [6]:
#Connect to SQLite in-memory database
conn = sqlite3.connect(":memory:")

In [7]:
try:
    #Load datasets
    reddit_df = load_csv(reddit_file)
    news_df = load_csv(news_file)
    youtube_df = load_csv(youtube_file)

    #Store in SQL if datasets exist
    if reddit_df is not None:
        reddit_df.to_sql("reddit", conn, if_exists="replace", index=False)
    if news_df is not None:
        news_df.to_sql("news", conn, if_exists="replace", index=False)
    if youtube_df is not None:
        youtube_df.to_sql("youtube", conn, if_exists="replace", index=False)

    #SQL Query to merge datasets
    query = """
    SELECT Comment AS text, Source FROM reddit
    UNION ALL
    SELECT Comment AS text, Source FROM news
    UNION ALL
    SELECT Comment AS text, Source FROM youtube;
    """

    #Execute query and save merged data
    merged_df = pd.read_sql(query, conn)
    merged_df.to_csv(output_file, index=False)

    print(f"✅ Merged dataset saved with {len(merged_df)} records to {output_file}!")

except sqlite3.Error as e:
    print(f"❌ SQLite error: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")
finally:
    if conn:
        conn.close()

✅ Merged dataset saved with 1598 records to ../data/merged/merged_data.csv!
